# Q06: Extending Context Window
**Topic:** LLM | **Difficulty:** Hard | **Time:** 30 min
**Sheet:** GrindLLM50

---

## Question
How can you extend the context window of a transformer model? (e.g., sliding window, position interpolation)

## Detailed Answer

### The Problem
Models are trained with a fixed context length (e.g., 4096 tokens). At inference, if you provide longer sequences, positional encodings go out-of-distribution → performance degrades.

### Methods

**1. Position Interpolation (PI)**
Scale positions to fit original range: $pos' = pos \times \frac{L_{train}}{L_{test}}$
- Simple, effective with short fine-tuning
- Used to extend LLaMA from 2K → 32K

**2. NTK-Aware Scaling**
Scale the base of RoPE: $\theta' = \theta \times \alpha^{d/(d-2)}$
- Better preserves high-frequency information than linear PI
- Dynamic NTK: automatically adjust based on sequence length

**3. YaRN (Yet another RoPE extensioN)**
Combines NTK scaling with attention scaling and temperature factors.
- State-of-the-art for RoPE extension
- Used in Mistral, many open-source models

**4. Sliding Window Attention**
Each layer only attends to the last W tokens.
- With L layers, effective receptive field = L × W tokens
- O(n·W) complexity vs O(n²)
- Used in Mistral (W=4096)

**5. ALiBi (Attention with Linear Biases)**
Add linear bias to attention scores based on token distance.
- No positional embedding — instead: $\text{softmax}(QK^T - m \cdot |i-j|)$
- Generalizes to longer sequences without any modification
- Used in BLOOM, MPT

**6. Ring Attention / Blockwise**
Distribute long sequences across multiple devices with ring communication.
- Enables million+ token contexts

### Comparison
| Method | Training Needed? | Extension Factor | Complexity |
|--------|-----------------|-----------------|------------|
| PI | Short fine-tune | 4-16x | O(1) change |
| NTK/YaRN | Optional | 4-16x | O(1) change |
| Sliding window | From scratch | Unlimited | O(n·W) |
| ALiBi | From scratch | Unlimited | O(n²) |
| Ring Attention | No | Hardware-limited | Distributed |

## Key Takeaways
- **YaRN/NTK scaling** is the go-to for extending RoPE-based models
- **Sliding window** is the cleanest architectural solution (Mistral)
- **ALiBi** generalizes well but requires training from scratch
- Modern LLMs combine multiple approaches: RoPE + sliding window + FlashAttention